In [ ]:
import os
from warnings import filters

import jax.numpy as jnp
import jax
from caskade import Param, forward
import numpy as np
from cosmographi.cosmology import Cosmology
from cosmographi.source.base import TransientSource
from cosmographi.utils import flux
from cosmographi.utils.constants import Mpc_to_cm
from typing import Any

# V. 1.0: No parameters
The following code shows the use of an AGN source that has a constant luminosity density for testing purposes.

In [55]:
class AGNSourcev1_0(TransientSource):
    """
    An AGN source model.

    Note: This is a simplified model that should not be used yet. 

    Parameters
    ---------- 
    name: str. 
        Optional. Name of the source.
    """
    cosmology: Cosmology
    name: str

    def __init__(self, cosmology: Cosmology = None, name: str = None, **kwargs) -> None:
        super().__init__(cosmology=cosmology, name=name, **kwargs)
    
    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        """
        Compute the luminosity density at time t. 

        Parameters
        ----------
        t: jnp.ndarray. Time array in seconds. Either (N,) or (N,1)

        Returns
        -------
        luminosity_density: jnp.ndarray. Luminosity density in erg/s/Hz.
        """
        if t.ndim == 1:
            t = t[:, None]
        density = jnp.full((len(t),1), 0.04)
        return jnp.concatenate((t, density), axis=1)
        


In [56]:
agn = AGNSourcev1_0(name="AGN1")
print(agn.luminosity_density(jnp.array([0.0, 0.5, 3.0, 5.0])))

[[0.   0.04]
 [0.5  0.04]
 [3.   0.04]
 [5.   0.04]]


#V.1.1: One parameter, luminosity if a function of time and said parameter
The following code shows an AGN whose luminosity is a function of time and a parameter of the AGN

In [57]:
from astropy.time import Time

class AGNSourcev1_1(TransientSource):
    """
    An AGN source model.

    Note: This is a simplified model that should not be used yet. 

    Parameters
    ---------- 
    name: str. 
        Optional. Name of the source.
    t0: 
        Param. Date of the initial obsevation in MJD format.
    lum_at_t0:
        Param. Luminosity density at t0 in erg/s/Hz.

    """
    cosmology: Cosmology
    name: str
    t0: Param
    lum_at_t0: Param

    def __init__(self, cosmology: Cosmology = None, name: str = None, t0: float = None, lum_at_t0: float = None, **kwargs) -> None:
        super().__init__(cosmology=cosmology, name=name, **kwargs)
        self.t0 = Param("t0", t0, shape=(), description="Date of the initial observation in MJD format")
        self.lum_at_t0 = Param("lum_at_t0", lum_at_t0, shape=(), description="Luminosity density at t0 in erg/s/Hz")

    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        """
        Compute the luminosity density at time t. 

        Parameters
        ----------
        t: jnp.ndarray. Time array in seconds. Either (N,) or (N,1)

        Returns
        -------
        luminosity_density: jnp.ndarray. Luminosity density in erg/s/Hz.
        """
        if t.ndim == 1:
            t = t[:, None]
        density = jnp.full((len(t),1), (self.lum_at_t0.value + 0.01 * (t - self.t0.value)))
        print("t:", t)
        print("density:", density)
        return jnp.concatenate((t, density), axis=1)

In [58]:
agn = AGNSourcev1_1(name="AGN2", t0=Time("2023-01-01").mjd, lum_at_t0=0.04)
print(agn.luminosity_density(jnp.array([0.0, 0.5, 3.0, 5.0])))

t: [[0. ]
 [0.5]
 [3. ]
 [5. ]]
density: [[-599.41 ]
 [-599.405]
 [-599.38 ]
 [-599.36 ]]
[[ 0.00000e+00 -5.99410e+02]
 [ 5.00000e-01 -5.99405e+02]
 [ 3.00000e+00 -5.99380e+02]
 [ 5.00000e+00 -5.99360e+02]]
